# Concentração financeira, Curva ABC, evolução temporal e clustering

In [ ]:
# Preparação automática apenas ao executar no Google Colab.
import os as _os
import csv as _csv
import sys as _sys
import subprocess as _subprocess
from pathlib import Path as _Path

try:
    import google.colab
    _executando_no_colab = True
except ImportError:
    _executando_no_colab = False

if _executando_no_colab:
    _pacotes = [
        "kagglehub>=1.0.2", "numpy>=1.26,<3", "pandas>=2.1,<3",
        "matplotlib>=3.8,<4", "scikit-learn>=1.4,<2",
        "statsmodels>=0.14,<1", "xgboost>=2,<4", "shap>=0.45,<1",
    ]
    _subprocess.run([_sys.executable, "-m", "pip", "install", "-q", *_pacotes], check=True)
    import kagglehub

    _caminho = _Path(kagglehub.dataset_download("belbino/indian-banking-transactions-20192024", path="indian_banking_transactions.csv"))
    if _caminho.is_dir():
        _caminho = _caminho / "indian_banking_transactions.csv"
    assert _caminho.is_file(), f"Dataset não encontrado: {_caminho}"
    with _caminho.open("r", encoding="utf-8-sig", newline="") as _arquivo:
        _campos = set(next(_csv.reader(_arquivo)))
    _esperados = set(["transaction_id","customer_id","transaction_date","account_type","transaction_amount","transaction_direction","merchant_category","transaction_status"])
    assert _esperados <= _campos, f"Colunas ausentes: {sorted(_esperados - _campos)}"
    _os.environ["FINANCE_BI_BANKING_CSV"] = str(_caminho)
    _os.environ.setdefault("FINANCE_BI_FAST_TEST", "1")
    _os.environ.setdefault("FINANCE_BI_OUTPUT_DIR", "/content/finance_bi_outputs")
    print("Dados Kaggle:", _caminho)
    print("Arquivo validado (cabeçalho):", len(_campos), "colunas")
    print("Modo rápido:", _os.environ["FINANCE_BI_FAST_TEST"] == "1")


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import RobustScaler

ROOT = Path(os.environ.get("FINANCE_BI_DATA_DIR", "data/raw"))
CSV = Path(os.environ.get("FINANCE_BI_BANKING_CSV", ROOT / "indian_banking_transactions.csv"))
OUT = Path(os.environ.get("FINANCE_BI_OUTPUT_DIR", "outputs")) / "01_concentracao"
OUT.mkdir(parents=True, exist_ok=True)
assert CSV.is_file(), f"CSV não encontrado: {CSV}"
cols = ["transaction_id", "customer_id", "transaction_date", "account_type", "transaction_amount", "transaction_direction", "merchant_category", "transaction_status"]
df = pd.read_csv(CSV, usecols=cols, parse_dates=["transaction_date"], low_memory=False)
df["transaction_amount"] = pd.to_numeric(df["transaction_amount"], errors="coerce")
df = df.dropna(subset=["customer_id", "transaction_date", "transaction_amount"]).drop_duplicates("transaction_id")
df = df.loc[(df["transaction_amount"] >= 0) & df["transaction_status"].eq("Success")].copy()
df["ano"] = df["transaction_date"].dt.year
assert not df.empty, "Nenhuma transação válida e concluída."
print("Transações válidas:", f"{len(df):,}", "| clientes:", df["customer_id"].nunique(), "|", df["transaction_date"].min().date(), "a", df["transaction_date"].max().date())
display(df.head())

In [ ]:
def concentracao_ano(parte):
    volume = parte.groupby("customer_id", observed=True)["transaction_amount"].sum().sort_values(ascending=False)
    total = float(volume.sum())
    if total == 0:
        return pd.DataFrame(), {"clientes": len(volume), "hhi": np.nan, "gini": np.nan, "top1pct": np.nan, "top10": np.nan}
    s = volume / total
    antes = s.cumsum().shift(fill_value=0)
    abc = np.select([antes < 0.80, antes < 0.95], ["A", "B"], default="C")
    tabela = pd.DataFrame({"customer_id": volume.index, "volume": volume.to_numpy(), "fracao": s.to_numpy(), "acumulado": s.cumsum().to_numpy(), "abc": abc})
    n = len(volume)
    v = np.sort(volume.to_numpy(dtype=float))
    gini = (2 * np.sum(np.arange(1, n+1) * v) / (n * v.sum()) - (n + 1) / n) if n > 0 else np.nan
    return tabela, {"clientes": n, "hhi": float(np.square(s).sum() * 10000), "gini": float(gini), "top1pct": float(s.iloc[:max(1,int(np.ceil(n*.01)))].sum()), "top10": float(s.iloc[:10].sum())}

series, indicadores = [], []
for ano, part in df.groupby("ano", observed=True):
    ab, metr = concentracao_ano(part)
    if not ab.empty:
        ab.insert(0, "ano", int(ano))
        series.append(ab)
    indicadores.append({"ano": int(ano), "volume_total_inr": float(part["transaction_amount"].sum()), "transacoes": len(part), **metr})
abc_por_ano = pd.concat(series, ignore_index=True)
concentracao = pd.DataFrame(indicadores).sort_values("ano")
print("Concentração anual (HHI: 0 a 10.000; maior indica concentração):")
display(concentracao)
print("Distribuição ABC por ano:")
display(abc_por_ano.groupby(["ano", "abc"], observed=True).agg(clientes=("customer_id", "size"), volume_inr=("volume", "sum")).reset_index())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
concentracao.plot(x="ano", y="hhi", marker="o", ax=axes[0], legend=False, title="Concentração HHI")
concentracao.plot(x="ano", y="top1pct", marker="o", ax=axes[1], legend=False, title="Participação do 1% de clientes")
concentracao.plot(x="ano", y="gini", marker="o", ax=axes[2], legend=False, title="Gini das movimentações")
for ax in axes: ax.grid(alpha=.2)
plt.tight_layout(); plt.show()
ultimo = abc_por_ano.loc[abc_por_ano["ano"].eq(abc_por_ano["ano"].max())]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(np.arange(1, len(ultimo)+1)/len(ultimo)*100, ultimo["acumulado"].to_numpy()*100)
ax.axhline(80, linestyle="--", color="gray"); ax.axhline(95, linestyle="--", color="gray")
ax.set(xlabel="Clientes ordenados por volume (% acumulado)", ylabel="Volume financeiro acumulado (%)", title="Curva ABC no último ano")
ax.grid(alpha=.25); plt.tight_layout(); plt.show()

In [ ]:
fim = df["transaction_date"].max()
carteira = df.groupby("customer_id", observed=True).agg(
    transacoes=("transaction_id","count"),
    valor_total=("transaction_amount","sum"),
    ticket_medio=("transaction_amount","mean"),
    desvio_valor=("transaction_amount","std"),
    ultima_data=("transaction_date","max"),
    primeira_data=("transaction_date","min"),
    meses_ativos=("transaction_date",lambda x: x.dt.to_period("M").nunique()),
    creditos=("transaction_direction",lambda x: x.eq("Credit").sum()),
).reset_index()
carteira["recencia_dias"] = (fim - carteira["ultima_data"]).dt.days
carteira["fracao_creditos"] = carteira["creditos"] / carteira["transacoes"]
carteira["desvio_valor"] = carteira["desvio_valor"].fillna(0)
features = ["transacoes", "valor_total", "ticket_medio", "desvio_valor", "recencia_dias", "meses_ativos", "fracao_creditos"]
X = carteira[features].copy()
for c in ["transacoes","valor_total","ticket_medio","desvio_valor","meses_ativos"]:
    X[c] = np.log1p(X[c].clip(lower=0))
scaler = RobustScaler()
Z = scaler.fit_transform(X)
if len(carteira) >= 10 and np.unique(Z,axis=0).shape[0] >= 3:
    scores = []
    for k in range(2, min(6, len(carteira)-1)+1):
        km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(Z)
        if 1 < len(set(km.labels_)) < len(km.labels_):
            scores.append((silhouette_score(Z, km.labels_, sample_size=min(2000,len(Z)), random_state=42), k))
    chosen_k = max(scores)[1] if scores else 2
    modelo = KMeans(n_clusters=chosen_k, random_state=42, n_init=10).fit(Z)
    carteira["cluster"] = modelo.labels_.astype(str)
else:
    carteira["cluster"] = "0"
print("Perfis descritivos (clusters calculados com toda a amostra; não são previsões):")
display(carteira.groupby("cluster").agg(clientes=("customer_id","count"), volume_total=("valor_total","sum"), ticket_mediano=("ticket_medio","median"), recencia_mediana=("recencia_dias","median"), transacoes_medianas=("transacoes","median")).sort_values("volume_total",ascending=False))
fig, ax=plt.subplots(figsize=(9,4))
carteira.boxplot(column="valor_total",by="cluster", ax=ax)
ax.set_yscale("symlog"); ax.set_title("Distribuição do volume por cluster"); plt.suptitle(""); plt.tight_layout();plt.show()

In [ ]:
abc_por_ano.to_csv(OUT / "curva_abc_por_ano.csv", index=False)
concentracao.to_csv(OUT / "indicadores_concentracao.csv", index=False)
carteira.drop(columns=["ultima_data", "primeira_data"]).to_csv(OUT / "segmentacao_clientes.csv", index=False)
print("Arquivos gerados:", *(str(p) for p in sorted(OUT.glob("*.csv"))), sep="\n")
print("Aviso: dados sintéticos; concentração transacional não mede risco financeiro ou exposição real ao crédito.")